In [1]:
import sys
sys.path.append('/home/azureuser/prathyusha/Kearney/prathyusha')
from utils import *
from pyspark.sql.functions import *
from pyspark.sql.window import Window
spark = instantiate_spark_sedona("10g")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/17 06:02:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/02/17 06:02:24 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in mesos/standalone/kubernetes and LOCAL_DIRS in YARN).
26/02/17 06:02:24 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/02/17 06:02:24 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/02/17 06:02:24 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/02/17 06:02:24 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.
26/02/17 06:02:24 WARN Utils: Service 'SparkUI' could not bind on port 4044. Attempting port 4045.


Spark Session and SedonaContext have been successfully initiated.


In [2]:
places_overture = spark.read.parquet('abfss://propheus-data-science@propheusdatabay.dfs.core.windows.net/Thailand/overture_final_places')
places_overture.printSchema()

root
 |-- overture_places_id: string (nullable = true)
 |-- geometry: string (nullable = true)
 |-- names: struct (nullable = true)
 |    |-- common: integer (nullable = true)
 |    |-- primary: string (nullable = true)
 |    |-- rules: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- between: integer (nullable = true)
 |    |    |    |-- language: string (nullable = true)
 |    |    |    |-- perspectives: integer (nullable = true)
 |    |    |    |-- side: integer (nullable = true)
 |    |    |    |-- value: string (nullable = true)
 |    |    |    |-- variant: string (nullable = true)
 |-- categories: struct (nullable = true)
 |    |-- alternate: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- primary: string (nullable = true)
 |-- socials: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- brand: struct (nullable = true)
 |    |-- names: struct (nullable = true)
 |    |  

In [3]:
import geohash as gh

def encode_geohash(lat, lon, precision = 8):
    if lat is None or lon is None:
        return None
    return gh.encode(lat, lon, precision)

geohash_udf = udf(encode_geohash, StringType())

In [4]:
places_overture_req = places_overture.select('overture_places_id', 'name', 'name_english', 'longitude', 'latitude', 'final_category')


places_overture_req = places_overture_req.withColumn("gh6", geohash_udf(col("latitude"), col("longitude"), lit(6)))
places_overture_req = places_overture_req.withColumn("gh5", geohash_udf(col("latitude"), col("longitude"), lit(5)))

# places_overture_req.show()

In [5]:
places_overture_req.show()

+--------------------+--------------------+--------------------+----------+---------+--------------------+------+-----+
|  overture_places_id|                name|        name_english| longitude| latitude|      final_category|   gh6|  gh5|
+--------------------+--------------------+--------------------+----------+---------+--------------------+------+-----+
|00003afd-99f7-48e...|         วัดหัวกุญแจ|Hua Khuan Key Temple|101.153929|13.249751|Civic and Social ...|w4rgmu|w4rgm|
|00009d96-a19b-444...|บริษัท เค.เอส.โกล...| K.S. Golf Co., Ltd.|100.520468|13.937006|            Services|w4rrr3|w4rrr|
|0003021f-3bb0-42c...|กลุ่มมังคุดชะอวดพ...|Mangosteen Group,...| 99.959742| 7.939569|      Grocery Stores|w1rj4t|w1rj4|
|000432d7-2f74-4d7...|      i blue cottage|      i blue cottage| 99.005908| 18.74725|              Hotels|w5q6su|w5q6s|
|0004cb8b-e54b-4ff...|      น้ำดื่มคัทลียา|Cutthaya Drinking...|103.970222|16.292226|              Retail|w6cutv|w6cut|
|0009d4a8-4b43-4de...|บ้านลุมพินีลาดปลา.

In [6]:
gh6_places = places_overture_req \
        .groupBy("gh6") \
        .pivot("final_category") \
        .agg(count("*")) \
        .fillna(0)

gh6_places.show(10, truncate=False)

+------+------------------------+----------------------------------+------------------+---------------------+------------------------------+-------------+---------+-------------+-------------------+--------------+----------+------+-------------------------------+------+---------+--------+--------------------------+--------------+
|gh6   |Amusement and Recreation|Auto and Gasoline Service Stations|Automotive Dealers|Children's Activities|Civic and Social Organizations|Eating Places|Education|Entertainment|Fashion and Apparel|Grocery Stores|Healthcare|Hotels|Industrial and Commercial Zones|Retail|Salon/Spa|Services|Sports and Fitness Centers|Transportation|
+------+------------------------+----------------------------------+------------------+---------------------+------------------------------+-------------+---------+-------------+-------------------+--------------+----------+------+-------------------------------+------+---------+--------+--------------------------+--------------+
|w6c

In [7]:
gh5_places = places_overture_req \
        .groupBy("gh5") \
        .pivot("final_category") \
        .agg(count("*")) \
        .fillna(0)

gh5_places.show(10, truncate=False)

+-----+------------------------+----------------------------------+------------------+---------------------+------------------------------+-------------+---------+-------------+-------------------+--------------+----------+------+-------------------------------+------+---------+--------+--------------------------+--------------+
|gh5  |Amusement and Recreation|Auto and Gasoline Service Stations|Automotive Dealers|Children's Activities|Civic and Social Organizations|Eating Places|Education|Entertainment|Fashion and Apparel|Grocery Stores|Healthcare|Hotels|Industrial and Commercial Zones|Retail|Salon/Spa|Services|Sports and Fitness Centers|Transportation|
+-----+------------------------+----------------------------------+------------------+---------------------+------------------------------+-------------+---------+-------------+-------------------+--------------+----------+------+-------------------------------+------+---------+--------+--------------------------+--------------+
|w5nz5|

In [8]:
gh5_places.count(), gh6_places.count()

(19218, 168895)

In [9]:
gh5_places.write.parquet(f'abfss://propheus-data-science@propheusdatabay.dfs.core.windows.net/Thailand/Model_Features_GH5_GH6_levels/gh5_places.csv')
gh6_places.write.parquet(f'abfss://propheus-data-science@propheusdatabay.dfs.core.windows.net/Thailand/Model_Features_GH5_GH6_levels/gh6_places.csv')